# Product Quantity Prediction

Notebook này chuyển bài toán từ dự đoán `Revenue` sang dự đoán `Quantity`.

Lý do: `Revenue = Quantity * Unit_Price`, nên dự đoán revenue trước khi biết quantity sẽ bị thiếu biến quan trọng nhất. Ở notebook này, `Quantity` là target chính; `Revenue` và `Profit` bị loại khỏi feature để tránh leakage.

## Modeling Framing

- **Target**: `Quantity`
- **Allowed input features**: date, geography, product/category fields, `Unit_Price`, and train-only historical quantity aggregates
- **Forbidden leakage features**: `Revenue`, `Profit`, and the current row's `Quantity`
- **Split**: time-based split, train before `2024-10-01`, test from `2024-10-01` onward

Because `Quantity` is a small, imbalanced count variable, the notebook evaluates both continuous regression metrics and rounded integer metrics.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, PoissonRegressor, Ridge
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "product_sales_dataset_final.csv"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
PLOT_DIR = PROJECT_ROOT / "reports" / "plots"

TARGET = "Quantity"
DATE_COLUMN = "Order_Date"
DATE_FORMAT = "%m-%d-%y"
TEST_START_DATE = pd.Timestamp("2024-10-01")
RANDOM_STATE = 42

BASE_CATEGORICAL_FEATURES = [
    "City",
    "State",
    "Region",
    "Category",
    "Sub_Category",
    "Product_Name",
]
TIME_FEATURES = ["year", "month", "quarter", "day_of_week", "is_weekend"]
QUANTITY_AGG_FEATURES = [
    "product_order_count",
    "product_avg_quantity",
    "category_avg_quantity",
    "sub_category_avg_quantity",
    "state_avg_quantity",
    "region_avg_quantity",
]
FORBIDDEN_FEATURES = {"Quantity", "Revenue", "Profit", "Order_ID", "Customer_Name", "Country"}

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

## Load and Validate Data

The formula check is kept as a data-quality validation only. `Revenue` is not used as a feature.

In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], format=DATE_FORMAT, errors="coerce")

required_columns = {
    "Order_ID",
    DATE_COLUMN,
    "Customer_Name",
    "City",
    "State",
    "Region",
    "Country",
    "Category",
    "Sub_Category",
    "Product_Name",
    "Quantity",
    "Unit_Price",
    "Revenue",
    "Profit",
}
missing_columns = sorted(required_columns.difference(df.columns))
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

formula_diff = (df["Revenue"].round(2) - (df["Quantity"] * df["Unit_Price"]).round(2)).abs()
checks = {
    "rows": len(df),
    "columns": df.shape[1],
    "missing_total": int(df.isna().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "date_parse_failures": int(df[DATE_COLUMN].isna().sum()),
    "date_min": str(df[DATE_COLUMN].min().date()),
    "date_max": str(df[DATE_COLUMN].max().date()),
    "revenue_formula_mismatch_rows": int((formula_diff > 0.01).sum()),
    "quantity_min": int(df[TARGET].min()),
    "quantity_max": int(df[TARGET].max()),
}
checks

## Quantity Distribution

`Quantity` is imbalanced. A strong-looking exact accuracy can be misleading because predicting `1` for every row already matches about half the rows.

In [ ]:
quantity_counts = df[TARGET].value_counts().sort_index()
quantity_share = (quantity_counts / len(df)).rename("share")
quantity_summary = pd.concat([quantity_counts.rename("orders"), quantity_share], axis=1)
display(quantity_summary)

afig, ax = plt.subplots(figsize=(8, 4))
quantity_counts.plot(kind="bar", ax=ax, color="#4c78a8")
ax.set_title("Quantity Distribution")
ax.set_xlabel("Quantity")
ax.set_ylabel("Order Count")
afig.tight_layout()
afig.savefig(PLOT_DIR / "quantity_distribution.png", dpi=160)
plt.show()

## Feature Engineering

Historical aggregate features are computed from the training window only, then mapped into both train and test. This avoids using future/test information.

In [ ]:
def add_time_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["year"] = out[DATE_COLUMN].dt.year
    out["month"] = out[DATE_COLUMN].dt.month
    out["quarter"] = out[DATE_COLUMN].dt.quarter
    out["day_of_week"] = out[DATE_COLUMN].dt.dayofweek
    out["is_weekend"] = out["day_of_week"].isin([5, 6]).astype(int)
    return out


def time_split(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train = frame[frame[DATE_COLUMN] < TEST_START_DATE].copy()
    test = frame[frame[DATE_COLUMN] >= TEST_START_DATE].copy()
    assert train[DATE_COLUMN].max() < test[DATE_COLUMN].min()
    return train, test


def add_quantity_aggregate_features(
    train_df: pd.DataFrame, test_df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, dict[Any, float]], dict[str, float]]:
    train = train_df.copy()
    test = test_df.copy()
    global_quantity_mean = float(train[TARGET].mean())

    specs = {
        "product_order_count": (
            "Product_Name",
            train.groupby("Product_Name").size(),
            float(train.groupby("Product_Name").size().median()),
        ),
        "product_avg_quantity": (
            "Product_Name",
            train.groupby("Product_Name")[TARGET].mean(),
            global_quantity_mean,
        ),
        "category_avg_quantity": (
            "Category",
            train.groupby("Category")[TARGET].mean(),
            global_quantity_mean,
        ),
        "sub_category_avg_quantity": (
            "Sub_Category",
            train.groupby("Sub_Category")[TARGET].mean(),
            global_quantity_mean,
        ),
        "state_avg_quantity": (
            "State",
            train.groupby("State")[TARGET].mean(),
            global_quantity_mean,
        ),
        "region_avg_quantity": (
            "Region",
            train.groupby("Region")[TARGET].mean(),
            global_quantity_mean,
        ),
    }

    mappings: dict[str, dict[Any, float]] = {}
    defaults: dict[str, float] = {}
    for feature_name, (source_col, series, default) in specs.items():
        mapping = series.to_dict()
        mappings[feature_name] = mapping
        defaults[feature_name] = float(default)
        train[feature_name] = train[source_col].map(mapping).fillna(default)
        test[feature_name] = test[source_col].map(mapping).fillna(default)

    return train, test, mappings, defaults


with_time = add_time_features(df)
train_df, test_df = time_split(with_time)
train_df, test_df, aggregate_mappings, aggregate_defaults = add_quantity_aggregate_features(train_df, test_df)

numeric_features = ["Unit_Price"] + TIME_FEATURES + QUANTITY_AGG_FEATURES
categorical_features = BASE_CATEGORICAL_FEATURES.copy()
feature_columns = numeric_features + categorical_features

assert FORBIDDEN_FEATURES.isdisjoint(feature_columns)
train_df.shape, test_df.shape, feature_columns

## Model Comparison

Regression models are evaluated on raw continuous predictions and rounded predictions. The rounded metrics answer: "If we need an integer quantity, how often are we exactly right or within one unit?"

In [ ]:
def make_one_hot_preprocessor() -> ColumnTransformer:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
    )


def make_ordinal_preprocessor() -> ColumnTransformer:
    numeric_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "ordinal",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                    encoded_missing_value=-1,
                ),
            ),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
    )


def quantity_metrics(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    rounded = np.rint(np.clip(y_pred, df[TARGET].min(), df[TARGET].max())).astype(int)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
        "MAE_rounded": float(mean_absolute_error(y_true, rounded)),
        "Exact_Accuracy": float(accuracy_score(y_true, rounded)),
        "Within_1_Accuracy": float((np.abs(y_true.to_numpy() - rounded) <= 1).mean()),
    }


models = {
    "Baseline_Mean": Pipeline(
        steps=[("preprocess", make_one_hot_preprocessor()), ("model", DummyRegressor(strategy="mean"))]
    ),
    "Baseline_Median": Pipeline(
        steps=[("preprocess", make_one_hot_preprocessor()), ("model", DummyRegressor(strategy="median"))]
    ),
    "Ridge": Pipeline(
        steps=[("preprocess", make_one_hot_preprocessor()), ("model", Ridge(alpha=2.0))]
    ),
    "Lasso": Pipeline(
        steps=[
            ("preprocess", make_one_hot_preprocessor()),
            ("model", Lasso(alpha=0.001, max_iter=10000, random_state=RANDOM_STATE)),
        ]
    ),
    "Poisson": Pipeline(
        steps=[
            ("preprocess", make_one_hot_preprocessor()),
            ("model", PoissonRegressor(alpha=0.01, max_iter=1000)),
        ]
    ),
    "HistGradientBoosting": Pipeline(
        steps=[
            ("preprocess", make_ordinal_preprocessor()),
            (
                "model",
                HistGradientBoostingRegressor(
                    max_iter=120,
                    learning_rate=0.05,
                    l2_regularization=0.05,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "RandomForest": Pipeline(
        steps=[
            ("preprocess", make_ordinal_preprocessor()),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=30,
                    max_depth=12,
                    min_samples_leaf=80,
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
        ]
    ),
}

In [ ]:
x_train = train_df[feature_columns]
y_train = train_df[TARGET]
x_test = test_df[feature_columns]
y_test = test_df[TARGET]

results = []
fitted_models = {}
predictions = {}
for model_name, pipeline in models.items():
    print(f"Training {model_name}...")
    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)
    results.append({"model": model_name, **quantity_metrics(y_test, y_pred)})
    fitted_models[model_name] = pipeline
    predictions[model_name] = y_pred

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)

## Current Result Interpretation

On this dataset, the quantity model is only slightly better than a mean baseline on RMSE/R2. A median baseline that always predicts `1` can look good on exact accuracy because `Quantity=1` is the majority class.

That means available fields do not strongly explain whether a specific order buys 1, 2, 3, or more units. This is a data limitation, not just a model-selection issue.

In [ ]:
non_baseline = results_df[~results_df["model"].str.startswith("Baseline")]
best_row = non_baseline.sort_values("RMSE").iloc[0].to_dict()
best_model_name = best_row["model"]
best_model = fitted_models[best_model_name]
best_pred = predictions[best_model_name]

print(f"Selected non-baseline quantity model: {best_model_name}")
print(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in best_row.items()}, indent=2))

## Downstream Revenue Check

If the goal is still revenue, we can compute:

```text
Revenue_hat = Unit_Price * Quantity_hat
```

This is conceptually cleaner than predicting revenue directly while pretending quantity does not exist. It also makes the limitation visible: quantity is hard to predict from these fields.

In [ ]:
def regression_metrics(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
    }

quantity_hat = best_pred
quantity_hat_rounded = np.rint(np.clip(quantity_hat, df[TARGET].min(), df[TARGET].max()))
revenue_hat = test_df["Unit_Price"].to_numpy() * quantity_hat
revenue_hat_rounded = test_df["Unit_Price"].to_numpy() * quantity_hat_rounded

downstream_revenue_metrics = pd.DataFrame(
    [
        {"method": "Unit_Price * Quantity_hat", **regression_metrics(test_df["Revenue"], revenue_hat)},
        {
            "method": "Unit_Price * rounded_Quantity_hat",
            **regression_metrics(test_df["Revenue"], revenue_hat_rounded),
        },
    ]
)
display(downstream_revenue_metrics)

## Save Quantity Model Artifacts

Running this cell writes a standalone bundle for quantity prediction. It stores the model, feature columns, and train-only aggregate mappings needed for single-row prediction.

In [ ]:
quantity_metrics_payload = {
    "selected_model": best_row,
    "metrics": results_df.to_dict("records"),
    "downstream_revenue_metrics": downstream_revenue_metrics.to_dict("records"),
    "feature_columns": feature_columns,
    "target": TARGET,
    "forbidden_features": sorted(FORBIDDEN_FEATURES),
    "test_start_date": str(TEST_START_DATE.date()),
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "leakage_note": "Quantity model excludes Revenue and Profit; aggregate features are computed from train data only.",
}

(ARTIFACT_DIR / "quantity_metrics.json").write_text(
    json.dumps(quantity_metrics_payload, indent=2), encoding="utf-8"
)
joblib.dump(best_model, ARTIFACT_DIR / "quantity_best_model.joblib")
joblib.dump(
    {
        "model": best_model,
        "selected_model": best_row,
        "feature_columns": feature_columns,
        "aggregate_mappings": aggregate_mappings,
        "aggregate_defaults": aggregate_defaults,
        "date_column": DATE_COLUMN,
        "date_format": DATE_FORMAT,
        "target": TARGET,
        "forbidden_features": sorted(FORBIDDEN_FEATURES),
    },
    ARTIFACT_DIR / "quantity_model_bundle.joblib",
)

ARTIFACT_DIR / "quantity_model_bundle.joblib"

## Single-Record Prediction Helper

This helper mirrors the notebook's feature engineering for one raw order-like record.

In [ ]:
def prepare_quantity_input(raw: dict[str, Any], bundle: dict[str, Any]) -> pd.DataFrame:
    row = pd.DataFrame([raw])
    row[DATE_COLUMN] = pd.to_datetime(row[DATE_COLUMN], format=DATE_FORMAT, errors="coerce")
    if row[DATE_COLUMN].isna().any():
        raise ValueError(f"{DATE_COLUMN} must match {DATE_FORMAT}, e.g. 12-15-24")

    row = add_time_features(row)
    source_by_aggregate = {
        "product_order_count": "Product_Name",
        "product_avg_quantity": "Product_Name",
        "category_avg_quantity": "Category",
        "sub_category_avg_quantity": "Sub_Category",
        "state_avg_quantity": "State",
        "region_avg_quantity": "Region",
    }
    for feature_name, source_col in source_by_aggregate.items():
        mapping = bundle["aggregate_mappings"][feature_name]
        default = bundle["aggregate_defaults"][feature_name]
        row[feature_name] = row[source_col].map(mapping).fillna(default)

    missing = sorted(set(bundle["feature_columns"]).difference(row.columns))
    if missing:
        raise ValueError(f"Missing derived features: {missing}")
    return row[bundle["feature_columns"]]


sample_order = {
    "Order_Date": "12-15-24",
    "City": "Boston",
    "State": "Massachusetts",
    "Region": "East",
    "Category": "Electronics",
    "Sub_Category": "Laptops",
    "Product_Name": "MacBook Air",
    "Unit_Price": 999.99,
}

bundle = joblib.load(ARTIFACT_DIR / "quantity_model_bundle.joblib")
sample_features = prepare_quantity_input(sample_order, bundle)
sample_quantity_hat = float(bundle["model"].predict(sample_features)[0])
{
    "predicted_quantity_continuous": round(sample_quantity_hat, 3),
    "predicted_quantity_integer": int(np.rint(np.clip(sample_quantity_hat, df[TARGET].min(), df[TARGET].max()))),
    "revenue_estimate_from_quantity": round(sample_order["Unit_Price"] * sample_quantity_hat, 2),
}

## Conclusion

This notebook reframes the task correctly: first estimate `Quantity`, then derive revenue if needed. The result is still limited because the current dataset has weak pre-sale demand signals. Stronger improvement would require features such as inventory exposure, promotions, traffic, customer intent, cart context, marketing spend, or historical customer/product purchase behavior.